# Download Sentinel-2 images from ESA Copernicus service

**DO NOT RUN THIS NOTEBOOK DURING THE COURSE. The data is pre-downloaded for the course.**

YOU WOULD NEED [ESA CDSE ACCOUNT](https://dataspace.copernicus.eu/) FOR THIS.

The input data for object detection exercise are Sentinel-2 level 1C RGB images. These are available from ESA CDSE service. CDSE service is open to everybody, but requires [registration](https://documentation.dataspace.copernicus.eu/APIs/S3.html). 

In this notebooks:
* Define specific images, based on available label data.
* Find the images from CDSE STAC.
* Download the images in .jp2 format from CDSE S3 storage.
* Convert to .tif format required by `geo2ml`-library
* Delete the .jp2 images.

In [ ]:
import os
from pathlib import Path

# STAC search
import pystac_client

# Download from S3
import boto3

Set folders.

In [ ]:
base_folder = os.path.join(os.sep, 'scratch', 'project_462001167', 'students', os.environ.get('USER'), 'GeoML') #TODO
exercise_folder = os.path.join(base_folder, '09_object_detection') 
data_folder = os.path.join(exercise_folder, 'sentinel2')

Set STAC specs

In [ ]:
STAC_endpoint = "https://stac.dataspace.copernicus.eu/v1"
S3_endpoint = 'https://eodata.dataspace.copernicus.eu'
stac_collection_id = "sentinel-2-l1c"
stac_asset_name = 'TCI'

Set CDSE S3 download settings.

**NOTE, add your own CDSE S3 access and secret key.**

In [ ]:
os.environ["AWS_ACCESS_KEY_ID"] = "xxx"                            # Replace with your CDSE S3 access key
os.environ["AWS_SECRET_ACCESS_KEY"] = "xx"                         # Replace with your CDSE S3 secret key
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"
os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
os.environ["AWS_HTTPS"] = "YES"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"

List of needed Sentinel-2 1C images

In [ ]:
L1C_tiles=["S2A_MSIL1C_20220813T095601_N0510_R122_T34VEM_20240717T115958",
             "S2B_MSIL1C_20220626T095039_N0510_R079_T35VLG_20240620T013500",
             "S2A_MSIL1C_20220721T095041_N0510_R079_T34VEM_20240712T224506",
             "S2B_MSIL1C_20210714T100029_N0500_R122_T34VEN_20230224T120043",
             "S2A_MSIL1C_20220624T100041_N0510_R122_T34VEN_20240714T110124"]
L1C_tiles

Make sure that you are in the right folder. Add new folder for the images.

In [ ]:
os.chdir(exercise_folder)
if not os.path.isdir(data_folder):
    os.makedirs(data_folder)

Open connection to the CDSE STAC service.

In [ ]:
stac_catalog = pystac_client.Client.open(STAC_endpoint)

Open connection to the CDSE S3 service, where data is stored.

In [ ]:
s3_resource = boto3.resource('s3', endpoint_url=S3_endpoint)

Define function to find the image from CDSE service, download it and convert to Geotiff format.

In [ ]:
def download_asset_url_by_collection_and_id(collection, id, asset):
    # STAC search settings
    params = {
        "collections": collection,
        "filter": {"op": "=", "args": [{"property": "id"}, id]}
    }
    # Find item from STAC catalog
    item= list(stac_catalog.search(**params).items_as_dicts())
    #print(item)
    # Retrieve the URL to the asset. Remove the beginning of the URL for download.
    url = item[0]['assets'][asset]['href'].replace('s3://eodata/','')
    # Define local path for downloaded file,
    filename = os.path.join(data_folder, os.path.basename(url))
    # Download the file.
    s3_resource.Object('eodata', url).download_file(filename)
    # Define local path for Geotiff file.
    filename_tif = filename.replace('jp2','tif')
    # Convert to Geotiff using GDAL commandline 
    # Possible only in Jupyter, use Rasterio, if pure Python solution needed.
    !gdal_translate {filename} {filename_tif} -co "TILED=YES"
    # Remove the original .jp2 file.
    os.remove(filename)
    print('Downloaded: ' + filename_tif)

Download all files and save in Geotiff format.

In [ ]:
for tile in L1C_tiles:
    download_asset_url_by_collection_and_id(stac_collection_id, tile, stac_asset_name)